In [ ]:
import sys
import setuptools._distutils.version
sys.modules['distutils.version'] = setuptools._distutils.version

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException
import pandas as pd
import time
import os

# ── All category URLs to scrape (from sitemap) ────────────────────────────────
URLS = [
    "https://www.carousell.com.my/categories/women-s-fashion-4/?tab=marketplace",
    "https://www.carousell.com.my/categories/men-s-fashion-3/?tab=marketplace",
    "https://www.carousell.com.my/categories/beauty-personal-care-11/?tab=marketplace",
    "https://www.carousell.com.my/categories/luxury-20/?tab=marketplace",
    "https://www.carousell.com.my/categories/video-gaming-697/?tab=marketplace",
    "https://www.carousell.com.my/categories/audio-595/?tab=marketplace",
    "https://www.carousell.com.my/categories/photography-6/?tab=marketplace",
    "https://www.carousell.com.my/categories/furniture-home-living-13/?tab=marketplace",
    "https://www.carousell.com.my/categories/tv-home-appliances-6098/?tab=marketplace",
    "https://www.carousell.com.my/categories/babies-kids-19/?tab=marketplace",
    "https://www.carousell.com.my/categories/hobbies-toys-6245/?tab=marketplace",
    "https://www.carousell.com.my/categories/health-nutrition-6267/?tab=marketplace",
    "https://www.carousell.com.my/categories/sports-equipment-10/?tab=marketplace",
    "https://www.carousell.com.my/categories/snacks-1210/?tab=marketplace",
    "https://www.carousell.com.my/categories/auto-accessories-109/?tab=marketplace",
    "https://www.carousell.com.my/categories/motorbikes-108/?tab=marketplace",
    "https://www.carousell.com.my/categories/pet-supplies-29/?tab=marketplace",
    "https://www.carousell.com.my/categories/cars-32/?tab=marketplace",
    "https://www.carousell.com.my/categories/tickets-vouchers-18/?tab=marketplace",
    "https://www.carousell.com.my/categories/property-102/?tab=marketplace",
    "https://www.carousell.com.my/categories/services-1426/?tab=marketplace",
    "https://www.carousell.com.my/categories/jobs-1439/?tab=marketplace",
    "https://www.carousell.com.my/categories/everything-else-16/?tab=marketplace",
    "https://www.carousell.com.my/categories/free-items-2156/?tab=marketplace", # Pravin
]

# ── Config ─────────────────────────────────────────────────────────────────────
CSV_FILE        = "carousell_data_pravin.csv"
PROGRESS_FILE   = "carousell_progress_pravin.txt"
MAX_RECORDS     = 100_000
SAVE_EVERY      = 50
SCROLL_PAUSE    = 2.5
MAX_STALE       = 10   # fallback: stop after this many consecutive empty rounds even with button

# ── Browser setup ──────────────────────────────────────────────────────────────
options = uc.ChromeOptions()
# options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument(
    "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
)

driver = uc.Chrome(options=options, version_main=147)
print("✅ Browser started!")

# ── Resume: load existing CSV & completed-URL log ──────────────────────────────
results  = []
seen_ids = set()

if os.path.exists(CSV_FILE):
    print(f"📂 Found existing file – resuming from '{CSV_FILE}'…")
    df_old   = pd.read_csv(CSV_FILE)
    results  = df_old.to_dict(orient="records")
    if "listing_id" in df_old.columns:
        seen_ids = set(df_old["listing_id"].dropna().astype(str))
        print(f"   ↳ Resumed with {len(seen_ids)} previously scraped cards.")
    else:
        print("   ⚠️  Old CSV missing 'listing_id' – duplicates may occur.")
else:
    print("🚀 Starting fresh scrape.")

completed_urls = set()
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE, "r") as f:
        completed_urls = set(line.strip() for line in f if line.strip())
    print(f"   ↳ {len(completed_urls)} URLs already completed – will skip them.\n")


def save():
    pd.DataFrame(results).to_csv(CSV_FILE, index=False)
    print(f"   💾 Saved {len(results):,} records → {CSV_FILE}")


def mark_url_done(url):
    with open(PROGRESS_FILE, "a") as f:
        f.write(url + "\n")


def parse_cards():
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(driver.page_source, "html.parser")

    cards = soup.select(
        '[data-testid^="listing-card-"]'
        ':not([data-testid="listing-card-text-seller-name"])'
        ':not([data-testid="listing-card-btn-like"])'
    )

    new_records = []
    for card in cards:
        raw_testid = card.get("data-testid", "")
        listing_id = raw_testid.replace("listing-card-", "")

        if not listing_id or listing_id in seen_ids:
            continue

        img_el  = card.select_one('a[href^="/p/"] img')
        product = img_el["alt"].strip() if img_el and img_el.get("alt") else None

        link_el     = card.select_one('a[href^="/p/"]')
        listing_url = (
            "https://www.carousell.com.my" + link_el["href"].split("?")[0]
            if link_el else None
        )

        price_el = card.select_one("div.D_aYM p[title]")
        price    = price_el["title"].strip() if price_el else None

        condition = None
        price_div = card.select_one("div.D_aYM")
        if price_div:
            sib       = price_div.find_next_sibling("p")
            condition = sib.text.strip() if sib else None

        seller_el = card.select_one('[data-testid="listing-card-text-seller-name"]')
        seller    = seller_el.text.strip() if seller_el else None

        time_el     = card.select_one(".D_aZi p")
        time_posted = time_el.text.strip() if time_el else None

        like_el = card.select_one('[data-testid="listing-card-btn-like"] span')
        likes   = like_el.text.strip() if like_el else "0"

        if not product:
            continue

        seen_ids.add(listing_id)
        new_records.append({
            "listing_id":  listing_id,
            "product":     product,
            "price":       price,
            "condition":   condition,
            "seller":      seller,
            "time_posted": time_posted,
            "likes":       likes,
            "listing_url": listing_url,
            "source_url":  driver.current_url,
        })

    return new_records


def find_show_more_button():
    """
    Returns the 'Show more results' button element if it exists and is visible,
    otherwise returns None.
    """
    try:
        btn = driver.find_element(
            By.XPATH, "//button[contains(., 'Show more results')]"
        )
        return btn if btn.is_displayed() else None
    except Exception:
        return None


def scrape_url(url):
    """Load one category URL and scroll until no 'Show more' button or MAX_RECORDS reached."""
    global results

    print(f"\n{'='*70}")
    print(f"🌐 Scraping: {url}")
    print(f"   Progress: {len(results):,} total records so far")
    print(f"{'='*70}")

    driver.get(url)
    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, '[data-testid^="listing-card-"]')
            )
        )
        print("   ✅ Page loaded.")
    except TimeoutException:
        print("   ❌ Timed out waiting for cards – skipping this URL.")
        return

    time.sleep(2)

    stale_count     = 0
    last_save_count = len(results)

    while len(results) < MAX_RECORDS:

        # ── Parse whatever cards are currently in the DOM ──────────────────
        new_records = parse_cards()

        if new_records:
            results.extend(new_records)
            stale_count = 0
            print(f"   📊 Total: {len(results):,}  (+{len(new_records)} new)")

            if len(results) - last_save_count >= SAVE_EVERY:
                save()
                last_save_count = len(results)
        else:
            stale_count += 1
            print(f"   ⏳ No new cards ({stale_count}/{MAX_STALE})")

        # ── Hard stale fallback (keeps MAX_STALE as a safety net) ─────────
        if stale_count >= MAX_STALE:
            print("   ✅ Stale limit hit – moving to next URL.")
            break

        # ── Check for 'Show more results' button ──────────────────────────
        show_more = find_show_more_button()

        if show_more is None:
            # No button present → page is exhausted; skip to next URL immediately
            print("   🚫 'Show more results' not found – page exhausted, skipping URL.")
            break

        # Button found: scroll it into view and click
        driver.execute_script("arguments[0].scrollIntoView(true);", show_more)
        time.sleep(0.5)
        show_more.click()
        print("   👉 Clicked 'Show more results'")
        time.sleep(SCROLL_PAUSE)

        # Nudge scroll to trigger any lazy-loaded content
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(SCROLL_PAUSE)
        driver.execute_script("window.scrollBy(0, -300);")
        time.sleep(0.3)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(0.5)


# ── Main loop ─────────────────────────────────────────────────────────────────
urls_to_scrape = [u for u in URLS if u not in completed_urls]
print(f"\n📋 {len(urls_to_scrape)} URLs to scrape "
      f"({len(completed_urls)} already done).\n")

for i, url in enumerate(urls_to_scrape, 1):
    print(f"\n[{i}/{len(urls_to_scrape)}]", end="")
    scrape_url(url)
    mark_url_done(url)
    save()

    if len(results) >= MAX_RECORDS:
        print(f"\n🏁 Hit MAX_RECORDS ({MAX_RECORDS:,}) – stopping early.")
        break

    time.sleep(3)

# ── Final save ────────────────────────────────────────────────────────────────
driver.quit()
save()
print(f"\n🏁 Finished. {len(results):,} records saved to '{CSV_FILE}'.")